# Query Translation

Ce notebook implémente et compare 4 techniques de query translation pour améliorer le RAG.

- Multi-Query : Génère N reformulations parallèles + union des documents retrouvés 
- RAG Fusion : Multi-Query + Reciprocal Rank Fusion pour reranker les résultats 
- Step-Back : Reformule vers une question plus générale/abstraite pour mieux ancrer le contexte 
- Decomposition : Découpe la question en sous-questions traitées séquentiellement 


NB : 
- HyDE est implémenté dans hyde.ipynb
- La réécriture de requête simple est dans RAG.ipynb

In [49]:
import warnings
warnings.filterwarnings('ignore')

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.load import dumps, loads
from operator import itemgetter

In [50]:
from api_config import configure_google_api_key
GOOGLE_API_KEY = configure_google_api_key()

## Chargement de la base vectorielle et du LLM

In [51]:
CHROMA_DIR = "db/chroma_minecraft_multivec"
STORE_DIR = "db/local_chunks_store"

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    task_type="retrieval_document"
)

vectorstore = Chroma(
    collection_name="minecraft_summaries",
    persist_directory=CHROMA_DIR,
    embedding_function=gemini_embeddings
)

from langchain_classic.storage import LocalFileStore
from langchain_classic.retrievers import MultiVectorRetriever

fs_store = LocalFileStore(STORE_DIR)

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=fs_store,
    id_key="doc_id",
)

llm = ChatOllama(model="qwen3:8b", temperature=0)

print(f"Base vectorielle chargée : {vectorstore._collection.count()} chunks")

Base vectorielle chargée : 173 chunks


## Helpers

In [52]:
def format_docs(docs):
    """Formate une liste de Documents en texte lisible avec leur source."""
    parts = []
    for doc in docs:
        source = doc.metadata.get("source", "inconnu")
        parts.append(f"[{source}]\n{doc.page_content}")
    return "\n\n".join(parts)


def get_unique_union(lists_of_docs):
    """Déduplique une liste de listes de Documents en s'appuyant sur leur sérialisation JSON."""
    seen = set()
    unique = []
    for docs in lists_of_docs:
        for doc in docs:
            key = dumps(doc)
            if key not in seen:
                seen.add(key)
                unique.append(doc)
    return unique


# Prompt de réponse final partagé par plusieurs techniques
answer_prompt = ChatPromptTemplate.from_template("""
Tu es un expert du jeu Minecraft. Réponds en français à la question suivante en t'appuyant
uniquement sur les documents fournis. Si l'information est absente, dis-le clairement.

Contexte :
{context}

Question : {question}
""")

## Technique 1 : Multi-Query

Le LLM génère N reformulations de la question originale depuis des angles différents.  
Chaque reformulation est envoyée au retriever, et l'union dédupliquée des documents est utilisée comme contexte.

In [53]:
multi_query_prompt = ChatPromptTemplate.from_template("""
Tu es un assistant expert Minecraft. Ta tâche est de générer 5 reformulations différentes
de la question ci-dessous afin d'améliorer la recherche dans une base vectorielle.
Chaque reformulation doit aborder la question sous un angle légèrement différent
(synonymes, niveau d'abstraction, point de vue). 
Renvoie uniquement les 5 questions, une par ligne, sans numérotation ni commentaire.

Question originale : {question}
""")

generate_multi_queries = (
    multi_query_prompt
    | llm
    | StrOutputParser()
    | (lambda x: [q.strip() for q in x.strip().split("\n") if q.strip()])
)

multi_query_retrieval_chain = generate_multi_queries | retriever.map() | get_unique_union

multi_query_rag_chain = (
    {"context": multi_query_retrieval_chain | format_docs,
     "question": itemgetter("question")}
    | answer_prompt
    | llm
    | StrOutputParser()
)

In [54]:
question_test = "Quel est l'ingrédient de base indispensable pour l'alchimie ?"

print("Reformulations générées par Multi-Query \n")
reformulations = generate_multi_queries.invoke({"question": question_test})
for r in reformulations:
    print(" -", r)

print(f"\nDocuments récupérés \n")
docs_multi = multi_query_retrieval_chain.invoke({"question": question_test})
print(f"{len(docs_multi)} documents récupérés")

print("\nRéponse Multi-Query \n")
reponse_multi = multi_query_rag_chain.invoke({"question": question_test})
print(reponse_multi)

Reformulations générées par Multi-Query 

 - Quel est le composant essentiel pour l'alchimie ?
 - Quelle est la matière première indispensable à l'alchimie ?
 - Quel élément fondamental est nécessaire pour l'alchimie ?
 - Quel est le matériau de base incontournable de l'alchimie ?
 - Quelle est la substance de base critique pour l'alchimie ?

Documents récupérés 

8 documents récupérés

Réponse Multi-Query 

L'ingrédient de base indispensable pour l'alchimie dans Minecraft est la **fiole d'eau**. Selon le document fourni, la fiole d'eau est la base de toutes les potions. Elle est obtenue en remplissant une fiole avec de l'eau provenant d'un chaudron, d'une source d'eau ou en pêchant. C'est donc cette fiole d'eau qui sert de fondement pour la création des potions.


In [56]:
rag_fusion_prompt = ChatPromptTemplate.from_template("""
Tu es un assistant expert Minecraft. Génère 4 requêtes de recherche différentes liées
à la question suivante, pour maximiser la couverture thématique dans une base vectorielle.
Renvoie uniquement les 4 requêtes, une par ligne, sans numérotation ni commentaire.

Question : {question}
""")

generate_fusion_queries = (
    rag_fusion_prompt
    | llm
    | StrOutputParser()
    | (lambda x: [q.strip() for q in x.strip().split("\n") if q.strip()])
)


def reciprocal_rank_fusion(results: list[list], k: int = 60):
    """Fusionne plusieurs listes de documents rankés via RRF. Retourne (doc, score) trié."""
    fused_scores = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            fused_scores[doc_str] = fused_scores.get(doc_str, 0) + 1 / (rank + k)
    return [
        (loads(doc_str), score)
        for doc_str, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]


rag_fusion_retrieval_chain = generate_fusion_queries | retriever.map() | reciprocal_rank_fusion

rag_fusion_rag_chain = (
    {"context": rag_fusion_retrieval_chain
               | (lambda ranked: format_docs([doc for doc, _ in ranked[:6]])),
     "question": itemgetter("question")}
    | answer_prompt
    | llm
    | StrOutputParser()
)

In [57]:
print("Requêtes générées par RAG Fusion \n")
fusion_queries = generate_fusion_queries.invoke({"question": question_test})
for q in fusion_queries:
    print(" -", q)

print("\nClassement RRF \n")
ranked_docs = rag_fusion_retrieval_chain.invoke({"question": question_test})
for i, (doc, score) in enumerate(ranked_docs[:6]):
    src = doc.metadata.get("source", "?")
    print(f"  [{i+1}] score={score:.4f} | {src} | {doc.page_content[:80]}...")

print("\nRéponse RAG Fusion \n")
reponse_fusion = rag_fusion_rag_chain.invoke({"question": question_test})
print(reponse_fusion)

Requêtes générées par RAG Fusion 

 - Quel est l'ingrédient de base indispensable pour l'alchimie ?
 - Quel est l'ingrédient essentiel pour l'alchimie ?
 - Quel est l'élément de base indispensable pour l'alchimie ?
 - Quel est l'ingrédient principal pour l'alchimie ?

Classement RRF 

  [1] score=0.0661 | https://minecraft.fandom.com/fr/wiki/Fabrication | Pour fabriquer quelque chose, le joueur doit déplacer les objets de son inventai...
  [2] score=0.0659 | https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire | Manger du porc cuit ou une soupe de champignons afin de faire remonter sa faim d...
  [3] score=0.0640 | https://minecraft.fandom.com/fr/wiki/Cuisson | La cuisson (nom anglais : smelting ) est la façon dont on peut cuisiner, faire f...
  [4] score=0.0323 | wikipedia:Minecraft | Le joueur, par défaut droitier, peut utiliser ses deux mains, mais seule la main...
  [5] score=0.0320 | https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions | Q: Comment jete


## Technique 3 : Step-Back

Avant de répondre à la question précise, on génère une question plus générale/abstraite (step-back).  
Le retriever est lancé sur les deux questions : la réponse s'appuie sur un contexte à la fois spécifique et large.

Exemple :
- Question : « Quels ingrédients faut-il pour brasser une Potion de Force II ? »
- Step-back : « Comment fonctionne le système d'alchimie dans Minecraft ? »

In [58]:
step_back_examples = [
    {
        "input":  "Quels ingrédients faut-il pour fabriquer une épée en diamant ?",
        "output": "Comment fonctionne le système de fabrication (crafting) dans Minecraft ?",
    },
    {
        "input":  "Comment invoquer le Wither ?",
        "output": "Quels sont les boss et mobs spéciaux que l'on peut invoquer dans Minecraft ?",
    },
    {
        "input":  "Quelle est la durée d'effet d'une Potion de Régénération ?",
        "output": "Comment fonctionnent les potions et leurs effets dans Minecraft ?",
    },
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai",    "{output}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=step_back_examples,
)

step_back_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Tu es un expert Minecraft. Reformule la question posée en une version plus générale "
     "et abstraite qui est plus facile à retrouver dans une encyclopédie ou un wiki. "
     "Voici quelques exemples :"),
    few_shot_prompt,
    ("human", "{question}"),
])

generate_step_back_query = step_back_prompt | llm | StrOutputParser()

step_back_response_prompt = ChatPromptTemplate.from_template("""
Tu es un expert Minecraft. Réponds en français à la question précise en t'appuyant
sur les deux contextes ci-dessous. Ignore les informations non pertinentes.

Contexte général (step-back) :
{step_back_context}

Contexte spécifique :
{normal_context}

Question : {question}
""")

step_back_chain = (
    {
        "normal_context":    RunnableLambda(lambda x: x["question"]) | retriever | format_docs,
        "step_back_context": generate_step_back_query | retriever | format_docs,
        "question":          lambda x: x["question"],
    }
    | step_back_response_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
print("Question step-back générée \n")
step_back_q = generate_step_back_query.invoke({"question": question_test})
print(" =>", step_back_q)

print("\nRéponse Step-Back \n")
reponse_step_back = step_back_chain.invoke({"question": question_test})
print(reponse_step_back)

Question step-back générée 




## Technique 4 : Decomposition

On décompose la question en sous-questions indépendantes que l'on traite séquentiellement.  
Chaque sous-réponse enrichit le contexte (Q&A pairs) disponible pour la suivante (answering recursively).
La dernière étape synthétise l'ensemble.

In [14]:
decomposition_prompt = ChatPromptTemplate.from_template("""
Tu es un assistant expert Minecraft. Décompose la question suivante en 3 sous-questions
plus simples et indépendantes qui, ensemble, permettent de répondre à la question principale.
Renvoie uniquement les 3 sous-questions, une par ligne, sans numérotation ni commentaire.

Question principale : {question}
""")

generate_subquestions = (
    decomposition_prompt
    | llm
    | StrOutputParser()
    | (lambda x: [q.strip() for q in x.strip().split("\n") if q.strip()])
)

# Prompt pour chaque sous-question (réponse récursive avec contexte accumulé)
sub_answer_prompt = ChatPromptTemplate.from_template("""
Tu es un expert Minecraft. Réponds en français à la sous-question en utilisant
le contexte fourni et les réponses aux questions précédentes si elles sont utiles.

Réponses aux questions précédentes :
{q_a_pairs}

Contexte extrait de la base de connaissances :
{context}

Sous-question : {question}
""")

# Prompt de synthèse finale
synthesis_prompt = ChatPromptTemplate.from_template("""
Tu es un expert Minecraft. En t'appuyant sur les paires question/réponse ci-dessous,
rédige une réponse synthétique et complète à la question principale.

Paires Q/R :
{context}

Question principale : {question}
""")


def run_decomposition_rag(question: str) -> str:
    """Exécute le RAG par décomposition avec réponse récursive."""
    subquestions = generate_subquestions.invoke({"question": question})
    print(f"  Sous-questions générées : {len(subquestions)}")
    for i, sq in enumerate(subquestions, 1):
        print(f"    {i}. {sq}")

    q_a_pairs = ""
    for sq in subquestions:
        docs = retriever.invoke(sq)
        context = format_docs(docs)
        answer = (
            sub_answer_prompt
            | llm
            | StrOutputParser()
        ).invoke({"question": sq, "context": context, "q_a_pairs": q_a_pairs})
        q_a_pairs += f"\n---\nQuestion : {sq}\nRéponse : {answer}\n"

    final_answer = (
        synthesis_prompt
        | llm
        | StrOutputParser()
    ).invoke({"context": q_a_pairs, "question": question})

    return final_answer

In [15]:
print("Réponse par Décomposition \n")
reponse_decomp = run_decomposition_rag(question_test)
print("\n" + reponse_decomp)

Réponse par Décomposition 



ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 27.320137673s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT


## Comparaison globale sur plusieurs questions

On évalue les 4 techniques sur des questions de différentes complexité :
- Factuelle simple : un fait direct dans la base
- Procédurale : une recette / fabrication en plusieurs étapes  
- Complexe : nécessite de combiner plusieurs informations

In [ ]:
questions_panel = [
    "Quel est l'ingrédient de base indispensable pour l'alchimie ?",
    "Quelles sont les différences entre les modes de jeu Survie et Créatif ?",
    "Comment se préparer efficacement pour combattre l'Ender Dragon ?",
]

In [ ]:
def run_all_techniques(question: str) -> dict:
    """Exécute les 4 techniques de query translation et retourne un dict de résultats."""
    print(f"\n{'='*60}")
    print(f"Question : {question}")
    print('='*60)

    results = {}

    print("\n1-Multi-Query\n")
    results["multi_query"] = multi_query_rag_chain.invoke({"question": question})
    print(results["multi_query"])

    print("\n2-RAG Fusion\n")
    results["rag_fusion"] = rag_fusion_rag_chain.invoke({"question": question})
    print(results["rag_fusion"])

    print("\n3-Step-Back\n")
    results["step_back"] = step_back_chain.invoke({"question": question})
    print(results["step_back"])

    print("\n4-Decomposition\n")
    results["decomposition"] = run_decomposition_rag(question)
    print(results["decomposition"])

    return results

In [ ]:
resultats = {}
for q in questions_panel:
    resultats[q] = run_all_techniques(q)

## 6. Conclusion

| Technique | Avantages | Limites | Coût (appels LLM) |
|---|---|---|---|
| Baseline | Rapide, simple | Peut rater des synonymes et paraphrases | 1 |
| Multi-Query | Meilleure couverture sémantique | Contexte plus bruité | 1 + 1 génération |
| RAG Fusion | Reranking plus robuste | Un peu plus lent | 1 + 1 génération |
| Step-Back | Meilleur pour questions très spécifiques | Peu utile | 1 + 1 génération |
| Decomposition | Meilleur pour questions complexes | Lent | N+1 |

- Questions simples / factuelles => Baseline ou Multi-Query
- Questions procédurales (craft, recettes) => RAG Fusion (bonne couverture + reranking)
- Questions complexes (stratégie, boss) => Decomposition
- Question très précise dans un domaine large => Step-Back

Donc pas mal de combiner HyDE + RAG Fusion : HyDE pour enrichir l'embedding de la requête, RAG Fusion pour consolider les résultats sur plusieurs sous-requêtes. 

In [7]:
# Test basique du retriever
docs = retriever.invoke("potion de force")
print(f"{len(docs)} docs trouvés")
if docs:
    print(docs[0].page_content[:200])

0 docs trouvés
